# Container terminal design: service level and costs

This example follows the seaside sizing workflow in *Ports and Waterways*, Part II, Chapter 4, Section 4.4 (printed pp. 136-142; PDF pp. 154-160 in the supplied 2026 PDF): throughput, fleet mix, vessel calls, crane workload, berths and quay length. Numerical inputs are attributed to Tables 4.5, 4.6, 4.9 and 4.10; the reference design is Table 4.11. Explanations and the economic extension here are original. Consult the book for the complete terminal layout, yard and landside design.

**Two models are deliberately kept separate:**
1. Reconstruct the book's sizing convention and its 4-berth, 14-crane result.
2. Use a workload-consistent E2/E2/n simulation to screen berth/crane alternatives against a maximum mean waiting/service ratio, then select the least-cost qualifying alternative.

The second model is an extension, not a claim to reproduce the book's queue occupancies. All financial inputs are illustrative, not values supplied by the book.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import t
from IPython.display import display
import openqtsim
from openqtsim.mt_engine import create_tasks, run_tasks, statistics_worker

print(f'OpenQTSim {openqtsim.__version__}')

OpenQTSim v0.5.1


## 1. Throughput and fleet inputs from the worked example

The 40/30/30 split is a **share of TEU throughput**, not a share of vessel calls. Divide each class's TEU flow by its call size and round calls upwards, as in Table 4.6. Weight vessel dimensions by these call counts.

The 2,460,000 TEU is the **quay throughput** used in the book's seaside calculation. Do not double the transhipment share again. The crane calculation uses 30 lifts/hour times 2 TEU/lift = 60 TEU/hour/crane (Table 4.10), not the 1.6 TEU/box conversion used elsewhere in the book. Keep those quantities distinct.

In [2]:
throughput_teu = 2_460_000
peak_factor = 1.2
operating_hours = 8_592
lifts_per_crane_hour = 30
teu_per_lift = 2
mooring_hours = 1.0
unmooring_hours = 1.0
max_cranes_per_berth = 4
max_waiting_factor = 0.10  # E[waiting time] / E[service time]
berth_candidates = list(range(1, 7))
fleet = pd.DataFrame({
    'class': ['Post Panamax I', 'VLCS', 'ULCS'],
    'throughput share': [0.4, 0.3, 0.3],
    'call size (TEU)': [1_750, 3_250, 4_875],
    'length (m)': [300, 397, 400],
}).set_index('class')

nr_arr = 30_000  # customers per replication per stable design
nr_replications = 4
seed = 2024
client = None  # optionally supply an existing Dask Client

# Illustrative financial inputs, NOT taken from Ports and Waterways.
discount_rate = 0.05
quay_capex_per_m = 60_000.0  # EUR/m, including berth infrastructure
quay_life_years = 50
quay_maintenance_fraction = 0.01  # fraction of initial capital per year
crane_capex = 8_000_000.0  # EUR/crane
crane_life_years = 25
crane_annual_opex = 300_000.0  # EUR/crane/year
vessel_cost_per_hour = 1_500.0  # EUR/vessel-hour, waiting AND in service

In [3]:
if not np.isclose(fleet['throughput share'].sum(), 1):
    raise ValueError('Fleet throughput shares must sum to one.')
positive = [throughput_teu, peak_factor, operating_hours, lifts_per_crane_hour, teu_per_lift,
            max_waiting_factor, quay_capex_per_m, quay_life_years, crane_capex,
            crane_life_years, vessel_cost_per_hour]
if not all(np.isfinite(value) and value > 0 for value in positive):
    raise ValueError('Throughput, productivity, service target, lifetimes and capital costs must be finite and positive.')
if min(mooring_hours, unmooring_hours, discount_rate, quay_maintenance_fraction, crane_annual_opex) < 0:
    raise ValueError('Times, discount rate and annual operating costs cannot be negative.')
if nr_replications < 2 or nr_arr <= 0 or max_cranes_per_berth < 1:
    raise ValueError('Use positive run lengths and crane slots, and at least two replications.')
if not berth_candidates or any(not isinstance(n, int) or n < 1 for n in berth_candidates):
    raise ValueError('Supply positive integer berth candidates.')
if (fleet[['call size (TEU)', 'length (m)', 'throughput share']] <= 0).any().any():
    raise ValueError('Fleet sizes, lengths and shares must be positive.')
fleet['annual flow (TEU)'] = throughput_teu * fleet['throughput share']
fleet['annual calls'] = np.ceil(fleet['annual flow (TEU)'] / fleet['call size (TEU)']).astype(int)
annual_calls = int(fleet['annual calls'].sum())
fleet['call share'] = fleet['annual calls'] / annual_calls
average_length = float((fleet['length (m)'] * fleet['call share']).sum())
largest_length = float(fleet['length (m)'].max())
average_exchange_teu = throughput_teu / annual_calls
crane_productivity = lifts_per_crane_hour * teu_per_lift
overhead = mooring_hours + unmooring_hours
peak_workload_hours = throughput_teu * peak_factor / crane_productivity
display(fleet.round(4))
print(f'{annual_calls} annual calls; mean vessel length {average_length:.2f} m')
print(f'Peak-adjusted handling workload: {peak_workload_hours:,.0f} single-crane hours/year')

,throughput share,call size (TEU),length (m),annual flow (TEU),annual calls,call share
class,,,,,,
Post Panamax I,0.4,1750,300,984000.0,563,0.5970
VLCS,0.3,3250,397,738000.0,228,0.2418
ULCS,0.3,4875,400,738000.0,152,0.1612


943 annual calls; mean vessel length 339.57 m
Peak-adjusted handling workload: 49,200 single-crane hours/year


## 2. Reconstruct the book's berth/crane progression

Starting with one berth, fill its four crane slots before adding the next berth. For n berths and k cranes the book's Table 4.11 occupancy column follows:

$$\rho_{book} = \frac{C p/(k P) + N t_m}{H}.$$

Here C is annual quay TEU, p the peak factor, P the crane productivity, N annual calls, and t_m total mooring/unmooring hours per call. There is **no division of the mooring term by n** in this reconstruction.

The small lookup below uses the 60-90% rows of the E2/E2/n queueing table (also used in OpenQTSim), with **linear interpolation**, not polynomial fitting or an M/M/n analytical formula. It is sufficient for the stable part of this particular book progression. Outside that range the lookup is marked unavailable rather than extrapolated. An occupancy at or above 100% is unstable.

Only numerical reference data are used here; the full tables and explanation remain in the book.

In [4]:
book_grid = np.array([0.6, 0.7, 0.8, 0.9])
book_factors = {
    1: [0.6306, 1.0391, 1.8653, 4.3590],
    2: [0.2222, 0.4125, 0.8300, 2.0000],
    3: [0.1103, 0.2275, 0.4600, 1.2000],
    4: [0.0639, 0.1441, 0.3300, 0.9200],
}
book_rows = []
for cranes in range(1, 17):
    berths = (cranes - 1) // max_cranes_per_berth + 1
    handling = peak_workload_hours / cranes
    occupancy = (handling + annual_calls * overhead) / operating_hours
    supported = berths in book_factors and book_grid[0] <= occupancy <= book_grid[-1]
    factor = float(np.interp(occupancy, book_grid, book_factors[berths])) if supported else np.nan
    book_rows.append({'berths': berths, 'cranes': cranes, 'handling hours': handling,
                      'book occupancy': occupancy, 'table waiting factor': factor,
                      'status': 'unstable' if occupancy >= 1 else ('interpolated' if supported else 'outside lookup range'),
                      'meets target': supported and factor <= max_waiting_factor})
    if book_rows[-1]['meets target']:
        break
book_path = pd.DataFrame(book_rows)
display(book_path.round(4))
book_feasible = book_path[book_path['meets target']]
if book_feasible.empty:
    raise ValueError('No book-path design passes within the reference lookup/search range; expand the reference model.')
book_design = book_feasible.iloc[0]
print(f'Book progression: {int(book_design["berths"])} berths and {int(book_design["cranes"])} cranes')

,berths,cranes,handling hours,book occupancy,table waiting factor,status,meets target
0,1,1,49200.0000,5.9458,NaN,unstable,False
1,1,2,24600.0000,3.0826,NaN,unstable,False
2,1,3,16400.0000,2.1283,NaN,unstable,False
3,1,4,12300.0000,1.6511,NaN,unstable,False
4,2,5,9840.0000,1.3648,NaN,unstable,False
5,2,6,8200.0000,1.1739,NaN,unstable,False
6,2,7,7028.5714,1.0375,NaN,unstable,False
7,2,8,6150.0000,0.9353,NaN,outside lookup range,False
8,3,9,5466.6667,0.8558,0.8726,interpolated,False
9,3,10,4920.0000,0.7921,0.4417,interpolated,False


Book progression: 4 berths and 14 cranes


## 3. A separate workload-consistent simulation model

For the extension, n identical berths each get an **effective average** k/n cranes. This can be fractional: it represents smoothed fleet productivity, **not** physically enforced simultaneous crane allocation. Each candidate has at least one crane per berth and at most four. No crane resource contention, vessel-specific crane limits or crane travel is simulated.

The fleet mix determines call frequency, mean exchange and quay length. The queue uses E2 interarrival and **aggregate E2 service times** with the derived mean; it does not sample vessel classes or claim the fleet mixture itself has an E2 distribution. This preserves a direct link with the book's E2/E2/n planning assumption. Mooring is included in the mean service time, not modelled as a separate fixed phase.

$$\lambda_{peak}=pN/H,\qquad E[ST]=\frac{C/N}{P(k/n)}+t_m,\qquad \rho=\lambda_{peak}E[ST]/n.$$

The peak factor is applied **once**, to the arrival rate. Do not also multiply the call size or mean handling time by it. Algebraically the simulation occupancy is

$$\rho_{sim}=Cp/(kPH)+pNt_m/(nH).$$

For the reference 4 berths/14 cranes this gives about **47.5%**, versus **62.9%** in the book convention. Without a peak adjustment to calls' mooring demand it would be about 46.4%; this notebook consistently applies peak traffic to both handling and mooring. Neither calculation should silently be substituted for the other.

Quay length follows the book's rule: one largest vessel, call-weighted average length for other berths, 15 m clearances and the 1.1 correction. It is used for **costing only**, not as an enforced length constraint in these queues.

In [5]:
peak_arrival_rate = peak_factor * annual_calls / operating_hours
rows = []
for n in berth_candidates:
    for k in range(n, max_cranes_per_berth * n + 1):
        mean_service = average_exchange_teu / (crane_productivity * (k / n)) + overhead
        occupancy = peak_arrival_rate * mean_service / n
        length = 30 + largest_length + 1.1 * (n - 1) * (15 + average_length)
        rows.append({'berths': n, 'cranes': k, 'effective cranes/berth': k / n,
                     'mean service (hours)': mean_service, 'occupancy': occupancy,
                     'stable': occupancy < 1, 'quay length (m)': length})
designs = pd.DataFrame(rows).set_index(['berths', 'cranes'])
stable_designs = designs[designs['stable']]
if stable_designs.empty:
    raise ValueError('No stable candidate: increase berth/crane capacity or reduce demand.')
display(designs.round(4))
print(f'{len(stable_designs)} stable designs; {len(stable_designs) * nr_replications} simulation tasks')
if (4, 14) in designs.index:
    print(f'Reference configuration simulation occupancy: {designs.loc[(4, 14), "occupancy"]:.2%}')

effective cranes/berth  mean service (hours)  occupancy  \
berths cranes                                                            
1      1                       1.0000               45.4783     5.9897   
       2                       2.0000               23.7391     3.1265   
       3                       3.0000               16.4928     2.1722   
       4                       4.0000               12.8696     1.6950   
2      2                       1.0000               45.4783     2.9948   
...                               ...                   ...        ...   
6      20                      3.3333               15.0435     0.3302   
       21                      3.5000               14.4224     0.3166   
       22                      3.6667               13.8577     0.3042   
       23                      3.8333               13.3422     0.2929   
       24                      4.0000               12.8696     0.2825   

               stable  quay length (m)  
berths cranes                           
1      1        False         430.0000  
       2        False         430.0000  
       3        False         430.0000  
       4        False         430.0000  
2      2        False         820.0287  
...               ...              ...  
6      20        True        2380.1437  
       21        True        2380.1437  
       22        True        2380.1437  
       23        True        2380.1437  
       24        True        2380.1437  

[69 rows x 5 columns]

51 stable designs; 204 simulation tasks
Reference configuration simulation occupancy: 47.49%


## 4. Simulate stable designs and assess service level

`nr_arr * nr_replications` is the simulated customer count **per stable design**, unrelated to the nominal annual call count. Each replication starts empty and drains to completion. Use longer runs near capacity; more replications alone do not remove start-up bias. The default grid can take several minutes serially; an existing Dask client can distribute tasks.

`create_tasks` takes **total system service capacity**, so `mu = n / E[ST]`. Candidate seed sequences are split again into separate arrival/service seeds per task. The estimated waiting factor is the replication mean Wq divided by the model's known mean service time. This is a ratio of means, not a mean of individual vessels' ratios.

For conservative screening, use an **approximate one-sided 95% Student-t upper bound** across replications. A design is labelled 'passes screening' only if that bound is below the target. This is not a simultaneous confidence guarantee across all designs, does not include start-up bias or parameter/model error, and is not a contractual service guarantee. Increase simulation effort around the shortlist before making a decision.

In [6]:
children = np.random.SeedSequence(seed).spawn(len(stable_designs))
tasks = []
task_designs = []
for ((n, k), design), child in zip(stable_designs.iterrows(), children):
    candidate_seed = int(child.generate_state(1)[0])
    candidate_tasks = create_tasks('E2', 'E2', [n], [design['occupancy']], nr_arr,
                                   mu=n / design['mean service (hours)'],
                                   nr_replications=nr_replications, seed=candidate_seed)
    tasks.extend(candidate_tasks)
    task_designs.extend([(n, k)] * len(candidate_tasks))
results = pd.DataFrame(run_tasks(tasks, client, func=statistics_worker))
results['berths'] = [n for n, k in task_designs]
results['cranes'] = [k for n, k in task_designs]
summary = results.groupby(['berths', 'cranes'])['W_q'].agg(['mean', 'std', 'count'])
designs['mean wait (hours)'] = summary['mean']
designs['wait standard error (hours)'] = summary['std'] / np.sqrt(summary['count'])
designs['waiting factor'] = designs['mean wait (hours)'] / designs['mean service (hours)']
designs['factor standard error'] = designs['wait standard error (hours)'] / designs['mean service (hours)']
critical_value = t.ppf(0.95, nr_replications - 1)
designs['factor upper bound'] = designs['waiting factor'] + critical_value * designs['factor standard error']
designs['service status'] = 'unstable'
designs.loc[designs['stable'], 'service status'] = 'fails mean target'
designs.loc[designs['waiting factor'] <= max_waiting_factor, 'service status'] = 'uncertain at target'
designs.loc[designs['factor upper bound'] <= max_waiting_factor, 'service status'] = 'passes screening'
display(designs[['occupancy', 'waiting factor', 'factor upper bound', 'service status']].round(4))

occupancy  waiting factor  factor upper bound    service status
berths cranes                                                                 
1      1          5.9897             NaN                 NaN          unstable
       2          3.1265             NaN                 NaN          unstable
       3          2.1722             NaN                 NaN          unstable
       4          1.6950             NaN                 NaN          unstable
2      2          2.9948             NaN                 NaN          unstable
...                  ...             ...                 ...               ...
6      20         0.3302          0.0005              0.0007  passes screening
       21         0.3166          0.0003              0.0004  passes screening
       22         0.3042          0.0003              0.0004  passes screening
       23         0.2929          0.0002              0.0003  passes screening
       24         0.2825          0.0002              0.0004  passes screening

[69 rows x 4 columns]

## 5. Add annual costs

Annualise capital with $CRF=i(1+i)^L/[(1+i)^L-1]$ (or 1/L for zero interest). Quay cost depends on length; crane cost on installed crane count. Avoid adding a separate berth cost that double-counts the quay investment.

Vessel cost includes **both waiting and service time** because increasing cranes changes handling time. Omitting service-time cost could undervalue crane investment. The nominal annual call count N is used for annual costs, but Wq is estimated at peak demand. Thus $N c_v(Wq_{peak}+E[ST])$ is an explicitly conservative **peak-delay annualisation**, not a simulation of seasonal annual performance. No additional peak factor is applied to annual costs.

These are simplified economic/system costs, not a terminal operator cash-flow or complete feasibility study. Yard, land, transport, dredging, revenue, vessel-specific costs and equipment interactions are outside scope.

In [7]:
def capital_recovery_factor(rate, years):
    """Return the annual capital recovery fraction for a constant discount rate."""
    return 1 / years if rate == 0 else rate / (1 - (1 + rate) ** (-years))

quay_annual_rate = capital_recovery_factor(discount_rate, quay_life_years) + quay_maintenance_fraction
crane_annual_cost = crane_capex * capital_recovery_factor(discount_rate, crane_life_years) + crane_annual_opex
designs['quay cost (EUR/year)'] = designs['quay length (m)'] * quay_capex_per_m * quay_annual_rate
designs['crane cost (EUR/year)'] = designs.index.get_level_values('cranes') * crane_annual_cost
designs['vessel service cost (EUR/year)'] = annual_calls * vessel_cost_per_hour * designs['mean service (hours)']
designs['vessel waiting cost (EUR/year)'] = annual_calls * vessel_cost_per_hour * designs['mean wait (hours)']
cost_columns = ['quay cost (EUR/year)', 'crane cost (EUR/year)',
                'vessel service cost (EUR/year)', 'vessel waiting cost (EUR/year)']
designs['total cost (EUR/year)'] = designs[cost_columns].sum(axis=1, min_count=len(cost_columns))
designs.loc[~designs['stable'], ['vessel waiting cost (EUR/year)', 'total cost (EUR/year)']] = np.inf
designs['cost standard error (EUR/year)'] = annual_calls * vessel_cost_per_hour * designs['wait standard error (hours)']
qualified = designs[designs['service status'] == 'passes screening']
if qualified.empty:
    raise ValueError('No design passes service screening. Expand capacity or increase simulation effort for uncertain designs; do not select a failing design.')
unconstrained_best = designs.loc[designs['stable'], 'total cost (EUR/year)'].idxmin()
selected = qualified['total cost (EUR/year)'].idxmin()
shortlist = qualified.sort_values('total cost (EUR/year)')
display(shortlist[['occupancy', 'mean service (hours)', 'waiting factor', 'factor upper bound'] + cost_columns + ['total cost (EUR/year)']].head(10).round(4))
print(f'Cost minimum without service constraint: {unconstrained_best[0]} berths, {unconstrained_best[1]} cranes')
print(f'Least-cost design passing screening: {selected[0]} berths, {selected[1]} cranes')
if selected[0] == max(berth_candidates) or selected[1] == selected[0] * max_cranes_per_berth:
    print('Selected design touches a berth/crane boundary: test a wider range or revised crane-slot assumptions before claiming a global optimum.')

occupancy  mean service (hours)  waiting factor  \
berths cranes                                                    
3      12         0.5650               12.8696          0.0829   
4      16         0.4237               12.8696          0.0114   
       15         0.4476               13.5942          0.0149   
       14         0.4749               14.4224          0.0190   
       13         0.5063               15.3779          0.0262   
       12         0.5430               16.4928          0.0371   
       11         0.5864               17.8103          0.0575   
5      19         0.3541               13.4416          0.0019   
       18         0.3708               14.0773          0.0024   
       20         0.3390               12.8696          0.0014   

               factor upper bound  quay cost (EUR/year)  \
berths cranes                                             
3      12                  0.0868          4.703014e+06   
4      16                  0.0122          6.218902e+06   
       15                  0.0159          6.218902e+06   
       14                  0.0208          6.218902e+06   
       13                  0.0280          6.218902e+06   
       12                  0.0385          6.218902e+06   
       11                  0.0595          6.218902e+06   
5      19                  0.0024          7.734789e+06   
       18                  0.0027          7.734789e+06   
       20                  0.0017          7.734789e+06   

               crane cost (EUR/year)  vessel service cost (EUR/year)  \
berths cranes                                                          
3      12               1.041144e+07                    1.820400e+07   
4      16               1.388191e+07                    1.820400e+07   
       15               1.301429e+07                    1.922900e+07   
       14               1.214668e+07                    2.040043e+07   
       13               1.127906e+07                    2.175208e+07   
       12               1.041144e+07                    2.332900e+07   
       11               9.543816e+06                    2.519264e+07   
5      19               1.648477e+07                    1.901321e+07   
       18               1.561715e+07                    1.991233e+07   
       20               1.735239e+07                    1.820400e+07   

               vessel waiting cost (EUR/year)  total cost (EUR/year)  
berths cranes                                                         
3      12                        1.509028e+06           3.482748e+07  
4      16                        2.073725e+05           3.851219e+07  
       15                        2.863894e+05           3.874859e+07  
       14                        3.882642e+05           3.915427e+07  
       13                        5.702097e+05           3.982024e+07  
       12                        8.659785e+05           4.082532e+07  
       11                        1.449740e+06           4.240509e+07  
5      19                        3.667015e+04           4.326944e+07  
       18                        4.867687e+04           4.331295e+07  
       20                        2.564416e+04           4.331683e+07

Cost minimum without service constraint: 3 berths, 12 cranes
Least-cost design passing screening: 3 berths, 12 cranes
Selected design touches a berth/crane boundary: test a wider range or revised crane-slot assumptions before claiming a global optimum.


## 6. Visualise the trade-off

First compare waiting/service factors with the 10% limit (or your edited target). Error bars extend upward to the one-sided screening bound. The reference 4/14 configuration is identified separately; it uses the **extension** occupancy here. The second chart shows the cost composition of the ten cheapest qualifying designs, with one-standard-error bars on total cost.

A low occupancy alone is not the objective. The chosen design must deliver acceptable service and balance quay, crane and vessel costs.

In [8]:
service_fig = go.Figure()
for n in berth_candidates:
    measured = designs.xs(n, level='berths')
    measured = measured[measured['stable']]
    service_fig.add_scatter(x=measured.index, y=measured['waiting factor'], mode='lines+markers',
                            name=f'{n} berths', error_y=dict(type='data', symmetric=False,
                            array=critical_value * measured['factor standard error'], arrayminus=np.zeros(len(measured))))
service_fig.add_hline(y=max_waiting_factor, line_dash='dash', annotation_text='Maximum mean waiting/service ratio')
if (4, 14) in designs.index and designs.loc[(4, 14), 'stable']:
    service_fig.add_scatter(x=[14], y=[designs.loc[(4, 14), 'waiting factor']], mode='markers',
                            marker=dict(symbol='diamond-open', size=16, color='black'), name='Book configuration in extension model')
service_fig.update_layout(title='Service-level screening: effective average crane allocation',
                          xaxis=dict(title='Installed STS cranes', dtick=1), yaxis_title='E[waiting] / E[service]',
                          template='plotly_white', height=600)
service_fig.show()

top = shortlist.head(10)
labels = [f'{n} berths / {k} cranes' for n, k in top.index]
cost_fig = go.Figure()
for column in cost_columns:
    cost_fig.add_bar(x=labels, y=top[column] / 1e6, name=column.replace(' (EUR/year)', ''))
cost_fig.add_scatter(x=labels, y=top['total cost (EUR/year)'] / 1e6, mode='markers',
                     marker=dict(color='black', symbol='diamond'), name='Total +/- 1 SE',
                     error_y=dict(type='data', array=top['cost standard error (EUR/year)'] / 1e6))
cost_fig.update_layout(title='Lowest-cost alternatives that pass service screening', barmode='stack',
                       yaxis_title='Annual cost (million EUR/year)', template='plotly_white', height=600)
cost_fig.show()

## 7. Interpret and refine

* The book progression stops at the first acceptable configuration along a particular sequence; it does not minimise cost across all berth/crane combinations.
* The extension changes the workload convention and screens a larger grid. A different answer can therefore result even before financial assumptions are changed. Do not attribute the difference solely to costs.
* Changing `max_waiting_factor` compares stricter/looser service requirements. It is a target for a ratio of means, not a promise for every ship or a probability of immediate service.
* Increasing throughput changes class flows, calls, handling times and arrival rates. Increasing cranes changes service time; increasing berths changes capacity, crane allocation and quay cost. Re-run all cells after edits.
* Run longer replications around the best alternatives, inspect uncertainty and expand candidate boundaries. The ranking is a simulation estimate, not proof of optimality.
* A more detailed follow-up would sample vessel classes and their call sizes, enforce integer crane availability and vessel limits, and simulate quay length and seasonal arrivals explicitly. That requires a different operational model, not just a new distribution label.

**Source:** *Ports and Waterways*, supplied 2026 PDF, Part II Chapter 4, Section 4.4, especially printed pp. 138-142 and Tables 4.5, 4.6, 4.9-4.12. The source credits those tables to TU Delft under CC BY-NC-SA 4.0. This notebook uses selected factual inputs and its own calculations; it does not reproduce the chapter text or illustrations.